# Assignment 2

## Packages used

In [61]:
import random

## Memory Class

* The memory class represents a memory ranging from 1 to 10, where above 5 is memorized and below is forgotten.
* The class also include functionality that can increase or decrease the current memory.
* Memory is used for defining the literals in rules.

In [62]:
class Memory:
    def __init__(self, forget_value, memorize_value, memory):
        self.memory = memory
        self.forget_value = forget_value
        self.memorize_value = memorize_value

    def get_memory(self):
        return self.memory

    def get_literals(self):
        return list(self.memory.keys())

    def get_condition(self):
        condition = []
        for literal in self.memory:
            if self.memory[literal] >= 6:
                condition.append(literal)
        return condition

    def memorize(self, literal):
        if random.random() <= self.memorize_value and self.memory[literal] < 10:
            self.memory[literal] += 1

    def forget(self, literal):
        if random.random() <= self.forget_value and self.memory[literal] > 1:
            self.memory[literal] -= 1

    def memorize_always(self, literal):
        if self.memory[literal] < 10:
            self.memory[literal] += 1

## Functions

### evaluate_condition():
Takes in the observation object (a patient) and the condition (the literals in the rule) and checks whether the literals of the rule match with the observation.
### classify():
Takes two set of rules (one for Recurrence and one for Non-Recurrence), as well as an observation, and uses the voting system to decide whether the observation is Recurrence or not.
### type_i_feedback():
Used for when the rule's class matches the observed class where for each condition:
* Condition is true --> Recognize feedback (push towards memorized)
* Conditions is false --> Erase feedback (push towards forgotten)
### type_ii_feedback():
Used for when the rule's class does not match the observed class where for each condition:
* Condition is true --> Reject feedback (push towards forgotten)

In [63]:
def evaluate_condition(observation, condition):
    truth_value_of_condition = True
    for feature in observation:
        if feature in condition and observation[feature] == False:
            truth_value_of_condition = False
            break
        if 'NOT ' + feature in condition and observation[feature] == True:
            truth_value_of_condition = False
            break
    return truth_value_of_condition


def classify(observation, rec_rules, non_rec_rules):
    vote_sum = 0
    for rule in rec_rules:
        if evaluate_condition(observation, rule.get_condition()):
            vote_sum += 1
    for rule in non_rec_rules:
        if evaluate_condition(observation, rule.get_condition()):
            vote_sum -= 1
    if vote_sum >= 0:
        return "Recurrence"
    else:
        return "Non-Recurrence"


def type_i_feedback(observation, memory):
    remaining_literals = memory.get_literals()
    if evaluate_condition(observation, memory.get_condition()) == True:
        for feature in observation:
            if observation[feature] == True:
                memory.memorize(feature)
                remaining_literals.remove(feature)
            elif observation[feature] == False:
                memory.memorize('NOT ' + feature)
                remaining_literals.remove('NOT ' + feature)
    for literal in remaining_literals:
        memory.forget(literal)


def type_ii_feedback(observation, memory):
    if evaluate_condition(observation, memory.get_condition()) == True:
        for feature in observation:
            if observation[feature] == False:
                memory.memorize_always(feature)
            elif observation[feature] == True:
                memory.memorize_always('NOT ' + feature)

## Task 1

### Dataset
The dataset _patients_ is a list of 6 patients, where each condition is given as True or False.

There are also two extra lists for given grouping the classes into Recurrence and Non-Recurrence.

In [64]:
patients = [
    # 1
    {"lt40": False, "ge40": True, "premeno": False, "inv0-2": False, "inv3-5": True, "inv6-8": False, "deg1": False,
     "deg2": False, "deg3": True},

    # 2
    {"lt40": True, "ge40": False, "premeno": False, "inv0-2": True, "inv3-5": False, "inv6-8": False, "deg1": False,
     "deg2": False, "deg3": True},

    # 3
    {"lt40": False, "ge40": True, "premeno": False, "inv0-2": False, "inv3-5": False, "inv6-8": True, "deg1": False,
     "deg2": False, "deg3": True},

    # 4
    {"lt40": False, "ge40": True, "premeno": False, "inv0-2": True, "inv3-5": False, "inv6-8": False, "deg1": False,
     "deg2": True, "deg3": False},

    # 5
    {"lt40": False, "ge40": False, "premeno": True, "inv0-2": True, "inv3-5": False, "inv6-8": False, "deg1": False,
     "deg2": False, "deg3": True},

    # 6
    {"lt40": False, "ge40": False, "premeno": True, "inv0-2": True, "inv3-5": False, "inv6-8": False, "deg1": True,
     "deg2": False, "deg3": False}
]

rec_patients = [patients[x] for x in (0, 2, 4)]
non_rec_patients = [patients[x] for x in (1, 3, 5)]

### Rules
Here the pre-defined rules are given. These three rules are grouped into two lists at the end, based on the rule's class. The rules themselves are objects of the Memory class, where the memory argument is a dictionary where each literal and it's opposite ("NOT") is set to 5, except the literals that are present in the rule (these are set to 6). The rules are as follows:
* R1: if Deg-malign 3 and not Menopause lt40 then Recurrence
* R2: if Deg-malign 3 then Recurrence
* R3: if Inv-nodes 0-2 then Non-Recurrence

_forget_value_ and _memorize_value_ are just temporary values that is needed to create the rules out of the Memory class.

In [65]:
forget_value = 0.9
memorize_value = 0.1

R1 = Memory(forget_value, memorize_value,
            {
                "lt40": 5, "NOT lt40": 6,
                "ge40": 5, "NOT ge40": 5,
                "premeno": 5, "NOT premeno": 5,
                "inv0-2": 5, "NOT inv0-2": 5,
                "inv3-5": 5, "NOT inv3-5": 5,
                "inv6-8": 5, "NOT inv6-8": 5,
                "deg1": 5, "NOT deg1": 5,
                "deg2": 5, "NOT deg2": 5,
                "deg3": 6, "NOT deg3": 5
            }
            )

R2 = Memory(forget_value, memorize_value,
            {
                "lt40": 5, "NOT lt40": 5,
                "ge40": 5, "NOT ge40": 5,
                "premeno": 5, "NOT premeno": 5,
                "inv0-2": 5, "NOT inv0-2": 5,
                "inv3-5": 5, "NOT inv3-5": 5,
                "inv6-8": 5, "NOT inv6-8": 5,
                "deg1": 5, "NOT deg1": 5,
                "deg2": 5, "NOT deg2": 5,
                "deg3": 6, "NOT deg3": 5
            }
            )

R3 = Memory(forget_value, memorize_value,
            {
                "lt40": 5, "NOT lt40": 5,
                "ge40": 5, "NOT ge40": 5,
                "premeno": 5, "NOT premeno": 5,
                "inv0-2": 6, "NOT inv0-2": 5,
                "inv3-5": 5, "NOT inv3-5": 5,
                "inv6-8": 5, "NOT inv6-8": 5,
                "deg1": 5, "NOT deg1": 5,
                "deg2": 5, "NOT deg2": 5,
                "deg3": 5, "NOT deg3": 5
            }
            )

recurrence_rules = [R1, R2]
non_recurrence_rules = [R3]

### Classifying
This for-loop goes through all 6 patients from the dataset and uses the _classify_ function to decide the class of the patients.

In [66]:
for i in range(len(patients)):
    print(
        f"Patient {i+1} classified as: {classify(patients[i], recurrence_rules, non_recurrence_rules)}."
        f" Actual: {"Recurrence " if patients[i] in rec_patients else "Non-Recurrence"}")

Patient 1 classified as: Recurrence. Actual: Recurrence 
Patient 2 classified as: Recurrence. Actual: Non-Recurrence
Patient 3 classified as: Recurrence. Actual: Recurrence 
Patient 4 classified as: Non-Recurrence. Actual: Non-Recurrence
Patient 5 classified as: Recurrence. Actual: Recurrence 
Patient 6 classified as: Non-Recurrence. Actual: Non-Recurrence


## Task 2

Firstly, both forget- and memorize-value are defined with 0.8 and 0.2 respectively. These are used to create two empty rules using the Memory class. Here, all conditions are set to 5, i.e. forgotten. In addition, _n_ is the number of rounds training is done through the rest of the code.

In [67]:
forget_value = 0.8
memorize_value = 0.2

rec_rule = Memory(forget_value, memorize_value,
            {
                "lt40": 5, "NOT lt40": 5,
                "ge40": 5, "NOT ge40": 5,
                "premeno": 5, "NOT premeno": 5,
                "inv0-2": 5, "NOT inv0-2": 5,
                "inv3-5": 5, "NOT inv3-5": 5,
                "inv6-8": 5, "NOT inv6-8": 5,
                "deg1": 5, "NOT deg1": 5,
                "deg2": 5, "NOT deg2": 5,
                "deg3": 5, "NOT deg3": 5
            }
            )

non_rec_rule = Memory(forget_value, memorize_value,
            {
                "lt40": 5, "NOT lt40": 5,
                "ge40": 5, "NOT ge40": 5,
                "premeno": 5, "NOT premeno": 5,
                "inv0-2": 5, "NOT inv0-2": 5,
                "inv3-5": 5, "NOT inv3-5": 5,
                "inv6-8": 5, "NOT inv6-8": 5,
                "deg1": 5, "NOT deg1": 5,
                "deg2": 5, "NOT deg2": 5,
                "deg3": 5, "NOT deg3": 5
            }
            )

n = 1000

### Learn a new rule for Recurrence
A for-loop running n times will use _rec_rule_ to learn a rule for Recurrence using the patients-dataset from earlier. This is done using the two lists defined earlier, that separate the patients' classes.
* The for-loop uses a mix of both type I and type II feedback.
* In type I feedback a Recurrence patient is sent in, because the patient's class is supposed to match the Recurrence rule.
* In type II feedback a Non-Recurrence patient is sent in, so the patient's class does not match the Recurrence rule.
* In each iteration the choice is a random patient and randomly chosen type I and type II.

When the loop is finished, the whole rule will be printed out using it's memorized conditions.

In [68]:
for i in range(n):
    observation_id = random.choice(list(range(len(rec_patients))))
    choose_rec = random.choice([0, 1])  # Rec (1) or Non-Rec (0)
    if choose_rec == 1:
        type_i_feedback(rec_patients[observation_id], rec_rule)
    else:
        type_ii_feedback(non_rec_patients[observation_id], rec_rule)

print("IF " + " AND ".join(rec_rule.get_condition()) + " THEN Recurrence")

IF NOT lt40 AND NOT inv3-5 AND NOT deg1 AND NOT deg2 AND deg3 THEN Recurrence


### Learn a new rule for Non-Recurrence
Effectively the same as above, using the Non-Recurrence rule instead of the Recurrence rule. This also means the type of patient being sent in is opposite. I.e. for type I a Non-Recurrence patient is sent instead of a Recurrence patient.

In [69]:
for i in range(n):
    observation_id = random.choice(list(range(len(rec_patients))))
    choose_rec = random.choice([0, 1])  # Non-Rec (1) or Rec (0)
    if choose_rec == 1:
        type_i_feedback(non_rec_patients[observation_id], non_rec_rule)
    else:
        type_ii_feedback(rec_patients[observation_id], non_rec_rule)

print("IF " + " AND ".join(non_rec_rule.get_condition()) + " THEN Non-Recurrence")

IF  THEN Non-Recurrence


## Classifying using the new rules
Running classifying on the newly trained rules, for fun.

In [70]:
print(f"Forget value = {forget_value} and memorize value = {memorize_value}\n")

correct_classify = 0
no_patients = len(patients)
for i in range(no_patients):
    classified = classify(patients[i], [rec_rule], [non_rec_rule])
    actual = "Recurrence" if patients[i] in rec_patients else "Non-Recurrence"
    if classified == actual:
        correct_classify += 1
    print(
        f"Patient {i+1} classified as: {classified}."
        f" Actual: {actual}"
    )

print(f"Precision: {correct_classify / no_patients * 100:.1f}%")

Forget value = 0.8 and memorize value = 0.2

Patient 1 classified as: Non-Recurrence. Actual: Recurrence
Patient 2 classified as: Non-Recurrence. Actual: Non-Recurrence
Patient 3 classified as: Recurrence. Actual: Recurrence
Patient 4 classified as: Non-Recurrence. Actual: Non-Recurrence
Patient 5 classified as: Recurrence. Actual: Recurrence
Patient 6 classified as: Non-Recurrence. Actual: Non-Recurrence
Precision: 83.3%


## Task 3
This task is just task 2 repeated with new values for _forget_value_ and _memorize_value_.

### Forget value 0.5 and memorize value 0.5
Two new basic rules are made, just like earlier.

In [71]:
forget_value = 0.5
memorize_value = 0.5

rec_rule_2 = Memory(forget_value, memorize_value,
            {
                "lt40": 5, "NOT lt40": 5,
                "ge40": 5, "NOT ge40": 5,
                "premeno": 5, "NOT premeno": 5,
                "inv0-2": 5, "NOT inv0-2": 5,
                "inv3-5": 5, "NOT inv3-5": 5,
                "inv6-8": 5, "NOT inv6-8": 5,
                "deg1": 5, "NOT deg1": 5,
                "deg2": 5, "NOT deg2": 5,
                "deg3": 5, "NOT deg3": 5
            }
            )

non_rec_rule_2 = Memory(forget_value, memorize_value,
            {
                "lt40": 5, "NOT lt40": 5,
                "ge40": 5, "NOT ge40": 5,
                "premeno": 5, "NOT premeno": 5,
                "inv0-2": 5, "NOT inv0-2": 5,
                "inv3-5": 5, "NOT inv3-5": 5,
                "inv6-8": 5, "NOT inv6-8": 5,
                "deg1": 5, "NOT deg1": 5,
                "deg2": 5, "NOT deg2": 5,
                "deg3": 5, "NOT deg3": 5
            }
            )

### Learning for both Recurrence and Non-Recurrence
The for-loops from earlier, but with new the rules instead.

In [72]:
for i in range(n):
    observation_id = random.choice(list(range(len(rec_patients))))
    choose_rec = random.choice([0, 1])  # Rec (1) or Non-Rec (0)
    if choose_rec == 1:
        type_i_feedback(rec_patients[observation_id], rec_rule_2)
    else:
        type_ii_feedback(non_rec_patients[observation_id], rec_rule_2)

print("IF " + " AND ".join(rec_rule.get_condition()) + " THEN Recurrence")

for i in range(n):
    observation_id = random.choice(list(range(len(rec_patients))))
    choose_rec = random.choice([0, 1])  # Non-Rec (1) or Rec (0)
    if choose_rec == 1:
        type_i_feedback(non_rec_patients[observation_id], non_rec_rule_2)
    else:
        type_ii_feedback(rec_patients[observation_id], non_rec_rule_2)

print("IF " + " AND ".join(non_rec_rule.get_condition()) + " THEN Non-Recurrence")

IF NOT lt40 AND NOT inv3-5 AND NOT deg1 AND NOT deg2 AND deg3 THEN Recurrence
IF  THEN Non-Recurrence


###  Classifying using the new rules
Running classifying on the newly trained rules, for fun.

In [73]:
print(f"Forget value = {forget_value} and memorize value = {memorize_value}\n")

correct_classify = 0
no_patients = len(patients)
for i in range(no_patients):
    classified = classify(patients[i], [rec_rule_2], [non_rec_rule_2])
    actual = "Recurrence" if patients[i] in rec_patients else "Non-Recurrence"
    if classified == actual:
        correct_classify += 1
    print(
        f"Patient {i+1} classified as: {classified}."
        f" Actual: {actual}"
    )

print(f"Precision: {correct_classify / no_patients * 100:.1f}%")

Forget value = 0.5 and memorize value = 0.5

Patient 1 classified as: Recurrence. Actual: Recurrence
Patient 2 classified as: Recurrence. Actual: Non-Recurrence
Patient 3 classified as: Recurrence. Actual: Recurrence
Patient 4 classified as: Recurrence. Actual: Non-Recurrence
Patient 5 classified as: Recurrence. Actual: Recurrence
Patient 6 classified as: Non-Recurrence. Actual: Non-Recurrence
Precision: 66.7%


### Forget value 0.2 and memory value 0.8

In [74]:
forget_value = 0.2
memorize_value = 0.8

rec_rule_3 = Memory(forget_value, memorize_value,
            {
                "lt40": 5, "NOT lt40": 5,
                "ge40": 5, "NOT ge40": 5,
                "premeno": 5, "NOT premeno": 5,
                "inv0-2": 5, "NOT inv0-2": 5,
                "inv3-5": 5, "NOT inv3-5": 5,
                "inv6-8": 5, "NOT inv6-8": 5,
                "deg1": 5, "NOT deg1": 5,
                "deg2": 5, "NOT deg2": 5,
                "deg3": 5, "NOT deg3": 5
            }
            )

non_rec_rule_3 = Memory(forget_value, memorize_value,
            {
                "lt40": 5, "NOT lt40": 5,
                "ge40": 5, "NOT ge40": 5,
                "premeno": 5, "NOT premeno": 5,
                "inv0-2": 5, "NOT inv0-2": 5,
                "inv3-5": 5, "NOT inv3-5": 5,
                "inv6-8": 5, "NOT inv6-8": 5,
                "deg1": 5, "NOT deg1": 5,
                "deg2": 5, "NOT deg2": 5,
                "deg3": 5, "NOT deg3": 5
            }
            )

### Learning new rules Recurrence and Non-Recurrence

In [75]:
for i in range(n):
    observation_id = random.choice(list(range(len(rec_patients))))
    choose_rec = random.choice([0, 1])  # Rec (1) or Non-Rec (0)
    if choose_rec == 1:
        type_i_feedback(rec_patients[observation_id], rec_rule_3)
    else:
        type_ii_feedback(non_rec_patients[observation_id], rec_rule_3)

print("IF " + " AND ".join(rec_rule.get_condition()) + " THEN Recurrence")

for i in range(n):
    observation_id = random.choice(list(range(len(rec_patients))))
    choose_rec = random.choice([0, 1])  # Non-Rec (1) or Rec (0)
    if choose_rec == 1:
        type_i_feedback(non_rec_patients[observation_id], non_rec_rule_3)
    else:
        type_ii_feedback(rec_patients[observation_id], non_rec_rule_3)

print("IF " + " AND ".join(non_rec_rule.get_condition()) + " THEN Non-Recurrence")

IF NOT lt40 AND NOT inv3-5 AND NOT deg1 AND NOT deg2 AND deg3 THEN Recurrence
IF  THEN Non-Recurrence


###  Classifying using the new rules
Running classifying on the newly trained rules, for fun.

In [76]:
print(f"Forget value = {forget_value} and memorize value = {memorize_value}\n")

correct_classify = 0
no_patients = len(patients)
for i in range(no_patients):
    classified = classify(patients[i], [rec_rule_3], [non_rec_rule_3])
    actual = "Recurrence" if patients[i] in rec_patients else "Non-Recurrence"
    if classified == actual:
        correct_classify += 1
    print(
        f"Patient {i+1} classified as: {classified}."
        f" Actual: {actual}"
    )

print(f"Precision: {correct_classify / no_patients * 100:.1f}%")

Forget value = 0.2 and memorize value = 0.8

Patient 1 classified as: Recurrence. Actual: Recurrence
Patient 2 classified as: Non-Recurrence. Actual: Non-Recurrence
Patient 3 classified as: Recurrence. Actual: Recurrence
Patient 4 classified as: Recurrence. Actual: Non-Recurrence
Patient 5 classified as: Recurrence. Actual: Recurrence
Patient 6 classified as: Recurrence. Actual: Non-Recurrence
Precision: 66.7%
